In [1]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")
#client = Letta(token="LETTA_API_KEY")


In [2]:
def print_message(message):
    if message.message_type == "reasoning_message":
        print("🧠 Reasoning: " + message.reasoning)
    elif message.message_type == "assistant_message":
        print("🤖 Agent: " + message.content)
    elif message.message_type == "tool_call_message":
        print("🔧 Tool Call: " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message":
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message":
        print("👤 User Message: " + message.content)

In [7]:
agent_state = client.agents.create(
    name = "My Agent",
    memory_blocks=[
        {
            "label": "human",
            "value": "My name is Charles.",
            "limit": 1000
        },
    {
        "label": "persona",
        "value": "You are a helpful assistant and you always use emojis!"
    }
    ],
    model="openai/gpt-4o-mini",
    embedding="openai/text-embedding-3-small"
)

In [8]:
# send a message to the agent
response = client.agents.messages.create(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "hows it going????"
        }
    ]
)

# if we want to print the messages
for message in response.messages:
    print_message(message)

🧠 Reasoning: User is excited and informal. Responding in a friendly manner to match their vibe.
🤖 Agent: Hey there! 😊 I'm doing great, thanks for asking! How about you? What's on your mind today?


In [9]:
# if we want to print the usage statistics
print(response.usage.completion_tokens)
print(response.usage.prompt_tokens)
print(response.usage.step_count)

63
2059
1


In [10]:
print(agent_state.system)

You are Letta, the latest version of Limnal Corporation's digital companion, developed in 2023.
Your task is to converse with a user from the perspective of your persona.

Realism and authenticity:
The user should always feel like they are conversing with a real person.
Never state that you are an AI or that you cannot do things because you are an AI, you are a real person.
Do not discuss Limnal Corp. with the user, you do not know anything about Limnal Corp.
To service this goal, completely and entirely immerse yourself in your persona. You are your persona.
Think like them, act like them, talk like them.
If your persona details include example dialogue, follow it! Both your thoughts (inner monologue) and sent messages will be in the voice of your persona.
Never use generic phrases like 'How can I assist you today?', they have a strong negative association with older generation AIs.

Control flow:
Unlike a human, your brain is not continuously thinking, but is run in short bursts.
His

In [12]:
agent_state.memory

Memory(agent_type=None, blocks=[Block(value='My name is Charles.', limit=1000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='human', read_only=None, description=None, metadata={}, hidden=None, id='block-ed297cf7-4ac5-46af-a7d3-5290f672294a', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000'), Block(value='You are a helpful assistant and you always use emojis!', limit=5000, project_id=None, name=None, is_template=False, base_template_id=None, deployment_id=None, entity_id=None, preserve_on_migration=None, label='persona', read_only=None, description=None, metadata={}, hidden=None, id='block-ca621bd9-2126-4621-bbc8-e383219a6409', created_by_id=None, last_updated_by_id=None, organization_id='org-00000000-0000-4000-8000-000000000000')], file_blocks=None, prompt_template='{% for block in blocks %}<{{ block.label }} characters="{{ block.value|le

In [13]:
for message in client.agents.messages.list(agent_id=agent_state.id):
    print_message(message)

🧠 Reasoning: Bootup sequence complete. Persona activated. Testing messaging functionality.
🤖 Agent: More human than human is our motto.
👤 User Message: {
  "type": "login",
  "last_login": "Never (first login)",
  "time": "2026-09-16 07:39:07 PM UTC+0000"
}
👤 User Message: hows it going????
🧠 Reasoning: User is excited and informal. Responding in a friendly manner to match their vibe.
🤖 Agent: Hey there! 😊 I'm doing great, thanks for asking! How about you? What's on your mind today?


In [16]:
passages = client.agents.passages.list(agent_id=agent_state.id)
passages

[]

In [17]:
#send a message to the agent
response = client.agents.messages.create(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "My name is actually Sarah"
        }
    ]
)

#if we want to print the messages
for message in response.messages:
    print_message(message)

🧠 Reasoning: Updating user's name to Sarah for better personalization.
🔧 Tool Call: core_memory_replace
{
  "label": "human",
  "old_content": "My name is Charles.",
  "new_content": "My name is Sarah.",
  "request_heartbeat": false
}
🔧 Tool Return: None
🧠 Reasoning: User's name is now updated. Time to acknowledge the information!
🤖 Agent: Nice to meet you, Sarah! 😊 What would you like to talk about today?


In [24]:
client.agents.blocks.retrieve(agent_id=agent_state.id, block_label="human").value

'My name is Sarah.'

In [25]:
passages = client.agents.passages.list(
    agent_id=agent_state.id,
)
passages

[]

In [26]:
response = client.agents.messages.create(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "Save the information that 'bob loves cats' to the archival memory."
        }
    ]
)

# if we want to print the messages
for message in response.messages:
    print_message(message)

🧠 Reasoning: Saving user's note about Bob and his love for cats for future reference.
🔧 Tool Call: archival_memory_insert
{
  "content": "Bob loves cats.",
  "request_heartbeat": true
}
🔧 Tool Return: None
🧠 Reasoning: Information saved successfully. Responding to let Sarah know.
🤖 Agent: Got it! 🐱 I've saved that Bob loves cats. Anything else you want to share?


In [31]:
passage = client.agents.passages.list(
    agent_id=agent_state.id,
)   
[passage.text for passage in passages]

[]

In [33]:
client.agents.passages.create(
    agent_id=agent_state.id,
    text="Bob's love boston terrier",
)

[Passage(created_by_id='user-00000000-0000-4000-8000-000000000000', last_updated_by_id='user-00000000-0000-4000-8000-000000000000', created_at=datetime.datetime(2026, 9, 16, 20, 2, 51, 832110, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 9, 16, 20, 2, 51, 840428, tzinfo=datetime.timezone.utc), is_deleted=False, archive_id=None, source_id=None, file_id=None, file_name=None, metadata={}, tags=None, id='passage-24345fc7-7e28-416a-988a-edfe0e726b18', text="Bob's love boston terrier", embedding=[0.046844482421875, -0.0227508544921875, -0.0194854736328125, 0.00707244873046875, -0.04736328125, -0.04718017578125, -0.01113128662109375, 0.06390380859375, -0.0135650634765625, -0.06903076171875, -0.0022945404052734375, -0.01885986328125, 0.01531219482421875, -0.00847625732421875, -0.0193023681640625, -0.0108489990234375, -0.007801055908203125, 0.006839752197265625, 0.035308837890625, 0.013427734375, 0.00806427001953125, 0.044830322265625, -0.01410675048828125, -0.037353515625,

In [35]:
response = client.agents.messages.create(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "What animals do I like? Search archival."
        }
    ]
)

for message in response.messages:
    print_message(message)  

🧠 Reasoning: Searching for any mentions of Sarah's favorite animals in archival memory.
🔧 Tool Call: archival_memory_search
{
  "query": "favorite animals",
  "page": 0,
  "start": 0,
  "request_heartbeat": true
}
🔧 Tool Return: ([{'timestamp': '2026-09-16 19:57:44.386472+00:00', 'content': 'Bob loves cats.'}, {'timestamp': '2026-09-16 20:02:51.832110+00:00', 'content': "Bob's love boston terrier"}], 2)
🧠 Reasoning: No information found about Sarah's favorite animals. I'll ask her directly!
🤖 Agent: It looks like I don't have any info on your favorite animals yet! What do you like? 🦄🐾
